In [2]:
"""
=========================================================
FASE B — CLASIFICACIÓN POR PACIENTE (folds fijos + votación)
=========================================================
Parte del fichero 'dataset_60_topado.csv' generado en la Fase A.

Qué hace:
  1. Carga el dataset de 60 pacientes (ya topado y con su fold).
  2. Validación cruzada de 10 folds FIJOS (sin aleatoriedad):
     en cada vuelta, 1 fold (6 pacientes, uno por clase) es
     validación y los otros 9 folds (54 pacientes) entrenan.
  3. Dentro de cada fold: escalado + SMOTE solo en entrenamiento.
  4. Clasifica las VENTANAS de validación con el modelo.
  5. AGREGA POR PACIENTE: junta las predicciones de las ventanas
     de cada paciente y decide UN diagnóstico por paciente por
     votación mayoritaria.
  6. Construye la matriz de confusión a nivel de PACIENTE (60 pac.)
     y guarda qué predijo para cada paciente (para el análisis de
     errores que piden las tutoras).

CÓMO USARLO:
  Cambia la ruta 'path_dataset' a la de tu ordenador y ejecuta.
=========================================================
"""

import pandas as pd
import numpy as np
from collections import Counter

from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# =========================================================
# RUTA — cámbiala a la de tu ordenador
# =========================================================
path_dataset = r"C:\Users\josem\Desktop\tfg\dataset_60_topado.csv"

# Parámetros del SVM (estos son los que luego variarás en el punto 3)
SVM_C     = 10
SVM_GAMMA = 'scale'

# =========================================================
# 1. CARGAR EL DATASET DE 60 PACIENTES
# =========================================================
df = pd.read_csv(path_dataset, sep=';', decimal=',')

# Columnas que NO son features
meta_cols = ['archivo', 'diagnostico', 'ventana', 'pid', 'fold']
feature_cols = [c for c in df.columns if c not in meta_cols]

# Asegurar que las features son numéricas
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=feature_cols)

print(f"Ventanas totales: {len(df)}")
print(f"Pacientes: {df['pid'].nunique()}")
print(f"Features: {len(feature_cols)}")

# =========================================================
# 2. VALIDACIÓN CRUZADA CON FOLDS FIJOS
# =========================================================
# Guardaremos, para cada paciente de validación, su diagnóstico
# real y el que predice el modelo (por votación de sus ventanas).
pred_por_paciente = []   # lista de dicts: pid, real, predicho

folds = sorted(df['fold'].unique())   # [1,2,...,10]

for fold_val in folds:
    # ---- separar entrenamiento (otros folds) y validación (este fold) ----
    train = df[df['fold'] != fold_val]
    val   = df[df['fold'] == fold_val]

    X_tr = train[feature_cols].values
    y_tr = train['diagnostico'].values
    X_val = val[feature_cols].values
    y_val = val['diagnostico'].values

    # ---- escalado: aprende en train, aplica en val (sin fuga) ----
    sc = MinMaxScaler()
    X_tr = sc.fit_transform(X_tr)
    X_val = sc.transform(X_val)

    # ---- SMOTE solo en entrenamiento ----
    counter = Counter(y_tr)
    k = min(5, min(counter.values()) - 1)
    if k >= 1:
        smt = SMOTETomek(smote=SMOTE(k_neighbors=k, random_state=42),
                         random_state=42)
        X_tr, y_tr = smt.fit_resample(X_tr, y_tr)

    # ---- entrenar el SVM ----
    modelo = SVC(kernel='rbf', C=SVM_C, gamma=SVM_GAMMA,
                 class_weight='balanced', random_state=42)
    modelo.fit(X_tr, y_tr)

    # ---- predecir las ventanas de validación ----
    val = val.copy()
    val['pred_ventana'] = modelo.predict(X_val)

    # ---- AGREGAR POR PACIENTE: votación mayoritaria ----
    # Para cada paciente del fold de validación, miramos las
    # predicciones de TODAS sus ventanas y nos quedamos con la
    # clase más repetida (la moda).
    for paciente, grupo in val.groupby('pid'):
        votos = Counter(grupo['pred_ventana'])
        # la clase más votada (en empate, Counter devuelve la primera
        # que alcanzó ese máximo; es determinista para un mismo orden)
        diagnostico_predicho = votos.most_common(1)[0][0]
        diagnostico_real = grupo['diagnostico'].iloc[0]
        pred_por_paciente.append({
            'pid': paciente,
            'real': diagnostico_real,
            'predicho': diagnostico_predicho,
            'n_ventanas': len(grupo),
            'votos': dict(votos)
        })

# =========================================================
# 3. RESULTADOS A NIVEL DE PACIENTE
# =========================================================
res = pd.DataFrame(pred_por_paciente)

clases = ['ASTHMA', 'BRON', 'COPD', 'HEART FAILURE', 'NORMAL', 'PNEUMONIA']
acc = accuracy_score(res['real'], res['predicho'])
bal = balanced_accuracy_score(res['real'], res['predicho'])

print(f"\n{'='*55}")
print(f"RESULTADOS POR PACIENTE  (SVM C={SVM_C}, gamma={SVM_GAMMA})")
print(f"{'='*55}")
print(f"  Accuracy (pacientes):          {acc:.4f}")
print(f"  Balanced Accuracy (pacientes): {bal:.4f}")
print(f"\n{classification_report(res['real'], res['predicho'], labels=clases, zero_division=0)}")

print("Matriz de confusión (por PACIENTE):")
cm = confusion_matrix(res['real'], res['predicho'], labels=clases)
print(pd.DataFrame(cm, index=clases, columns=clases).to_string())

# =========================================================
# 4. GUARDAR EL DETALLE PACIENTE A PACIENTE
# (para el análisis de errores que piden las tutoras)
# =========================================================
res_ordenado = res.sort_values(['real', 'pid'])
res_ordenado.to_csv('resultado_por_paciente.csv', index=False, sep=';')
print("\nDetalle guardado en 'resultado_por_paciente.csv'")
print("(incluye, por paciente: real, predicho, nº ventanas y votos)")

# Mostrar los pacientes MAL clasificados (los que hay que investigar)
fallos = res[res['real'] != res['predicho']]
print(f"\n{'='*55}")
print(f"PACIENTES MAL CLASIFICADOS: {len(fallos)} de {len(res)}")
print(f"{'='*55}")
for _, fila in fallos.iterrows():
    print(f"  {fila['pid']:<14} real={fila['real']:<14} "
          f"predicho={fila['predicho']:<14} votos={fila['votos']}")

Ventanas totales: 956
Pacientes: 60
Features: 41

RESULTADOS POR PACIENTE  (SVM C=10, gamma=scale)
  Accuracy (pacientes):          0.5833
  Balanced Accuracy (pacientes): 0.5833

               precision    recall  f1-score   support

       ASTHMA       0.60      0.60      0.60        10
         BRON       0.50      0.50      0.50        10
         COPD       1.00      1.00      1.00        10
HEART FAILURE       0.60      0.60      0.60        10
       NORMAL       0.25      0.30      0.27        10
    PNEUMONIA       0.62      0.50      0.56        10

     accuracy                           0.58        60
    macro avg       0.60      0.58      0.59        60
 weighted avg       0.60      0.58      0.59        60

Matriz de confusión (por PACIENTE):
               ASTHMA  BRON  COPD  HEART FAILURE  NORMAL  PNEUMONIA
ASTHMA              6     0     0              2       2          0
BRON                0     5     0              1       2          2
COPD                0     0